# Galaxy10 DECals — Notebook de entrenamiento (NPZ)

Este notebook está organizado para cumplir con la rúbrica del sprint usando una **sola arquitectura** (`ResNet-18`), **AMP**, **manejo explícito del desbalance**, **early stopping** y **checkpoint por F1 Macro**.

## Qué hace este notebook
1. Carga un dataset `.npz` de forma **flexible**.
2. Hace EDA rápido de formas, `dtype` y distribución de clases.
3. Construye un split estratificado train/val.
4. Entrena una **ResNet-18 preentrenada** con `AdamW + CosineAnnealingLR`.
5. Monitorea **F1 Macro** y **Especificidad Macro**.
6. Guarda `best_model.pth`.
7. Deja lista la celda para el **test ciego** con `evaluate_check.py`.
8. Exporta `requirements.txt`.

> La rúbrica exige una arquitectura única, entrenamiento con AMP, `batch size >= 16`, mitigación explícita del desbalance, `early stopping` con paciencia mínima 5 y guardar el mejor checkpoint según **F1 Macro**, además de entregar `best_model.pth`, `requirements.txt` y el resultado del script oficial. fileciteturn4file0

In [ ]:
# Si estás en Google Colab, descomenta esta celda solo si hace falta instalar algo.
# !pip install -q torch torchvision scikit-learn matplotlib pillow

In [ ]:
# =========================
# 1. Imports y configuración
# =========================
import os
import random
import json
import math
import subprocess
from pathlib import Path
from collections import Counter

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    f1_score,
    confusion_matrix,
    classification_report,
    balanced_accuracy_score,
)

from torchvision import models, transforms

SEED = 2024
DATA_PATH = "galaxy10_train_val.npz"   # cambia esta ruta si tu archivo tiene otro nombre
IMAGE_SIZE = 224
BATCH_SIZE = 16                        # la rúbrica exige >=16
NUM_WORKERS = 2
MAX_EPOCHS = 20
PATIENCE = 5
LR = 3e-4
WEIGHT_DECAY = 1e-4
VAL_RATIO_WITHIN_PUBLIC = 0.111111     # 10/90 -> deja el público aprox. en 80/10 respecto al total original
USE_WEIGHTED_SAMPLER = True
USE_CLASS_WEIGHTS_IN_LOSS = True
LOSS_NAME = "focal"                    # opciones: "focal" o "ce"
LABEL_SMOOTHING = 0.0                  # usar 0.0 con focal; si cambias a CE puedes probar 0.05
FREEZE_BACKBONE_EPOCHS = 1             # 0 = fine-tuning total desde el inicio
TEAM_NAME = "Nombre del Equipo"

CLASS_NAMES = [
    "Rounded Smooth",
    "In-between Smooth",
    "Cigar-shaped Smooth",
    "Barred Spiral",
    "Unbarred Tight Spiral",
    "Unbarred Loose Spiral",
    "Edge-on w/ Bulge",
    "Edge-on w/o Bulge",
    "Merging",
    "Disturbed",
]

def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
amp_enabled = torch.cuda.is_available()

print(f"Dispositivo: {device}")
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## Nota sobre el split
La rúbrica habla de separar el `train_val` público en `train/val` y también indica que el instructor ya retiene el `blind_test`. Por eso aquí se hace un split estratificado **dentro** del `.npz` público. fileciteturn4file0

In [ ]:
# =====================================
# 2. Utilidades para cargar el .npz
# =====================================
def summarize_npz(npz_obj):
    summary = {}
    for k in npz_obj.files:
        arr = npz_obj[k]
        summary[k] = {
            "shape": arr.shape,
            "dtype": str(arr.dtype)
        }
    return summary

def to_int_labels(y):
    y = np.asarray(y)
    if y.ndim > 1:
        # one-hot o probabilidades por clase
        y = np.argmax(y, axis=1)
    return y.astype(np.int64)

def infer_images_and_labels(npz_path):
    data = np.load(npz_path, allow_pickle=True)
    print("Claves detectadas:", data.files)
    print(json.dumps(summarize_npz(data), indent=2))

    lower_map = {k.lower(): k for k in data.files}

    # Caso 1: el archivo ya tiene splits explícitos
    split_candidates = [
        ("x_train", "y_train", "x_val", "y_val"),
        ("images_train", "labels_train", "images_val", "labels_val"),
        ("train_images", "train_labels", "val_images", "val_labels"),
    ]

    for xtr, ytr, xva, yva in split_candidates:
        if all(k in lower_map for k in [xtr, ytr, xva, yva]):
            X_train = np.asarray(data[lower_map[xtr]])
            y_train = to_int_labels(data[lower_map[ytr]])
            X_val = np.asarray(data[lower_map[xva]])
            y_val = to_int_labels(data[lower_map[yva]])
            return {
                "has_predefined_split": True,
                "X_train": X_train,
                "y_train": y_train,
                "X_val": X_val,
                "y_val": y_val,
                "raw_npz": data,
            }

    # Caso 2: archivo único con imágenes + etiquetas
    image_keys = ["images", "x", "imgs", "data", "x_all", "features"]
    label_keys = ["ans", "labels", "y", "targets", "y_all"]

    img_key = next((lower_map[k] for k in image_keys if k in lower_map), None)
    lbl_key = next((lower_map[k] for k in label_keys if k in lower_map), None)

    if img_key is None or lbl_key is None:
        raise KeyError(
            "No pude inferir automáticamente las claves de imágenes y etiquetas.\n"
            "Revisa las claves mostradas arriba y ajusta la función infer_images_and_labels()."
        )

    X = np.asarray(data[img_key])
    y = to_int_labels(data[lbl_key])

    return {
        "has_predefined_split": False,
        "X": X,
        "y": y,
        "raw_npz": data,
    }

bundle = infer_images_and_labels(DATA_PATH)

In [ ]:
# =====================================
# 3. Estandarización básica del formato
# =====================================
def ensure_nhwc_uint8(X):
    X = np.asarray(X)

    # Si viene en NCHW -> NHWC
    if X.ndim == 4 and X.shape[1] in [1, 3] and X.shape[-1] not in [1, 3]:
        X = np.transpose(X, (0, 2, 3, 1))

    if X.ndim != 4:
        raise ValueError(f"Se esperaba un tensor de imágenes de 4 dimensiones, pero llegó {X.shape}")

    # Escalar si viene en [0,1]
    if np.issubdtype(X.dtype, np.floating):
        x_min, x_max = float(X.min()), float(X.max())
        if x_max <= 1.0:
            X = (X * 255.0).clip(0, 255).astype(np.uint8)
        else:
            X = X.clip(0, 255).astype(np.uint8)
    else:
        X = X.astype(np.uint8)

    # Si es grayscale, repetir canales
    if X.shape[-1] == 1:
        X = np.repeat(X, 3, axis=-1)

    if X.shape[-1] != 3:
        raise ValueError(f"Se esperaban 3 canales RGB, pero llegó shape={X.shape}")

    return X

if bundle["has_predefined_split"]:
    X_train = ensure_nhwc_uint8(bundle["X_train"])
    y_train = bundle["y_train"]
    X_val = ensure_nhwc_uint8(bundle["X_val"])
    y_val = bundle["y_val"]
else:
    X = ensure_nhwc_uint8(bundle["X"])
    y = bundle["y"]

    X_train, X_val, y_train, y_val = train_test_split(
        X,
        y,
        test_size=VAL_RATIO_WITHIN_PUBLIC,
        random_state=SEED,
        shuffle=True,
        stratify=y,
    )

print("Train:", X_train.shape, y_train.shape)
print("Val:  ", X_val.shape, y_val.shape)
print("dtype train:", X_train.dtype, y_train.dtype)

In [ ]:
# =========================
# 4. EDA rápido
# =========================
def plot_class_distribution(labels, title):
    counts = np.bincount(labels, minlength=len(CLASS_NAMES))
    fig, ax = plt.subplots(figsize=(12, 4))
    ax.bar(np.arange(len(counts)), counts)
    ax.set_xticks(np.arange(len(counts)))
    ax.set_xticklabels([str(i) for i in range(len(counts))], rotation=0)
    ax.set_title(title)
    ax.set_xlabel("Clase")
    ax.set_ylabel("Número de imágenes")
    plt.show()

def show_samples_per_class(X, y, samples_per_class=3):
    num_classes = len(CLASS_NAMES)
    fig, axes = plt.subplots(num_classes, samples_per_class, figsize=(samples_per_class * 3, num_classes * 2.2))
    if samples_per_class == 1:
        axes = np.expand_dims(axes, axis=1)

    for cls in range(num_classes):
        idxs = np.where(y == cls)[0]
        chosen = idxs[:samples_per_class] if len(idxs) >= samples_per_class else idxs

        for j in range(samples_per_class):
            ax = axes[cls, j]
            ax.axis("off")
            if j < len(chosen):
                ax.imshow(X[chosen[j]])
                if j == 0:
                    ax.set_title(f"{cls}: {CLASS_NAMES[cls]}", fontsize=9, loc="left")
            else:
                ax.set_facecolor("white")
    plt.tight_layout()
    plt.show()

plot_class_distribution(y_train, "Distribución de clases — Train")
plot_class_distribution(y_val, "Distribución de clases — Validation")
show_samples_per_class(X_train, y_train, samples_per_class=3)

In [ ]:
# ==========================================
# 5. Dataset, transforms y dataloaders
# ==========================================
imagenet_mean = [0.485, 0.456, 0.406]
imagenet_std  = [0.229, 0.224, 0.225]

train_tfms = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=90),
    transforms.ColorJitter(brightness=0.10, contrast=0.10, saturation=0.10, hue=0.02),
    transforms.ToTensor(),
    transforms.Normalize(mean=imagenet_mean, std=imagenet_std),
])

val_tfms = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=imagenet_mean, std=imagenet_std),
])

class GalaxyDataset(Dataset):
    def __init__(self, X, y, transform=None):
        self.X = X
        self.y = y.astype(np.int64)
        self.transform = transform

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        image = self.X[idx]
        label = int(self.y[idx])

        if self.transform is not None:
            image = self.transform(image)
        else:
            image = torch.tensor(image).permute(2, 0, 1).float() / 255.0

        return image, label

train_ds = GalaxyDataset(X_train, y_train, transform=train_tfms)
val_ds   = GalaxyDataset(X_val, y_val, transform=val_tfms)

train_counts = np.bincount(y_train, minlength=len(CLASS_NAMES))
class_weights_np = 1.0 / np.maximum(train_counts, 1)
class_weights_np = class_weights_np / class_weights_np.sum() * len(class_weights_np)

sample_weights = class_weights_np[y_train]
sampler = WeightedRandomSampler(
    weights=torch.DoubleTensor(sample_weights),
    num_samples=len(sample_weights),
    replacement=True,
)

train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=not USE_WEIGHTED_SAMPLER,
    sampler=sampler if USE_WEIGHTED_SAMPLER else None,
    num_workers=NUM_WORKERS,
    pin_memory=True,
)

val_loader = DataLoader(
    val_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
)

print("Class counts:", train_counts)
print("Class weights:", np.round(class_weights_np, 4))
print(f"Train batches: {len(train_loader)} | Val batches: {len(val_loader)}")

In [ ]:
# ==========================================
# 6. Métricas
# ==========================================
def macro_specificity(y_true, y_pred, num_classes):
    cm = confusion_matrix(y_true, y_pred, labels=list(range(num_classes)))
    specificities = []

    for c in range(num_classes):
        tp = cm[c, c]
        fp = cm[:, c].sum() - tp
        fn = cm[c, :].sum() - tp
        tn = cm.sum() - (tp + fp + fn)

        denom = tn + fp
        spec = tn / denom if denom > 0 else 0.0
        specificities.append(spec)

    return float(np.mean(specificities)), np.array(specificities)

@torch.no_grad()
def predict_loader(model, loader, device):
    model.eval()
    all_preds, all_targets = [], []

    for imgs, targets in loader:
        imgs = imgs.to(device, non_blocking=True)
        targets = targets.to(device, non_blocking=True)

        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=amp_enabled):
            logits = model(imgs)

        preds = torch.argmax(logits, dim=1)

        all_preds.extend(preds.cpu().numpy())
        all_targets.extend(targets.cpu().numpy())

    return np.array(all_targets), np.array(all_preds)

def evaluate_model(model, loader, device):
    y_true, y_pred = predict_loader(model, loader, device)
    f1_macro = f1_score(y_true, y_pred, average="macro")
    spec_macro, spec_per_class = macro_specificity(y_true, y_pred, num_classes=len(CLASS_NAMES))
    bal_acc = balanced_accuracy_score(y_true, y_pred)
    return {
        "f1_macro": float(f1_macro),
        "spec_macro": float(spec_macro),
        "bal_acc": float(bal_acc),
        "y_true": y_true,
        "y_pred": y_pred,
        "spec_per_class": spec_per_class,
    }

def plot_confusion(y_true, y_pred, normalize=False):
    cm = confusion_matrix(y_true, y_pred, labels=list(range(len(CLASS_NAMES))))
    if normalize:
        cm = cm.astype(np.float64) / np.maximum(cm.sum(axis=1, keepdims=True), 1)

    fig, ax = plt.subplots(figsize=(8, 7))
    im = ax.imshow(cm)
    plt.colorbar(im, ax=ax)
    ax.set_xticks(np.arange(len(CLASS_NAMES)))
    ax.set_yticks(np.arange(len(CLASS_NAMES)))
    ax.set_xticklabels(range(len(CLASS_NAMES)))
    ax.set_yticklabels(range(len(CLASS_NAMES)))
    ax.set_xlabel("Predicción")
    ax.set_ylabel("Real")
    ax.set_title("Matriz de confusión" + (" normalizada" if normalize else ""))
    plt.tight_layout()
    plt.show()

In [ ]:
# ==========================================
# 7. Modelo base: ResNet-18
# ==========================================
def build_model(num_classes=10):
    model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
    in_features = model.fc.in_features
    model.fc = nn.Linear(in_features, num_classes)
    return model

model = build_model(num_classes=len(CLASS_NAMES)).to(device)

def count_trainable_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Parámetros entrenables: {count_trainable_params(model):,}")

In [ ]:
# Verificación rápida de VRAM con batch real (si hay GPU)
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    dummy = torch.randn(BATCH_SIZE, 3, IMAGE_SIZE, IMAGE_SIZE, device=device)
    model.eval()
    with torch.no_grad():
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=True):
            _ = model(dummy)
    print(f"VRAM usada en forward de prueba: {torch.cuda.memory_allocated() / 1e9:.2f} GB")
else:
    print("No hay GPU activa; esta verificación se debe correr cuando estés en GPU.")

In [ ]:
# ==========================================
# 8. Pérdida, optimizador y scheduler
# ==========================================
class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0, weight=None):
        super().__init__()
        self.gamma = gamma
        self.weight = weight

    def forward(self, logits, targets):
        ce = F.cross_entropy(logits, targets, weight=self.weight, reduction="none")
        pt = torch.exp(-ce)
        return ((1.0 - pt) ** self.gamma * ce).mean()

class_weights = torch.tensor(class_weights_np, dtype=torch.float32, device=device) if USE_CLASS_WEIGHTS_IN_LOSS else None

if LOSS_NAME.lower() == "focal":
    criterion = FocalLoss(gamma=2.0, weight=class_weights)
else:
    criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=LABEL_SMOOTHING)

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=MAX_EPOCHS)

scaler = torch.amp.GradScaler("cuda", enabled=amp_enabled)

criterion

In [ ]:
# ==========================================
# 9. Freeze temporal del backbone (opcional)
# ==========================================
def set_backbone_trainable(model, trainable: bool):
    for name, param in model.named_parameters():
        if not name.startswith("fc."):
            param.requires_grad = trainable

if FREEZE_BACKBONE_EPOCHS > 0:
    set_backbone_trainable(model, trainable=False)
    print("Backbone congelado temporalmente.")
else:
    print("Fine-tuning total desde la época 1.")

In [ ]:
# ==========================================
# 10. Entrenamiento
# ==========================================
history = {
    "train_loss": [],
    "val_f1": [],
    "val_spec": [],
    "val_bal_acc": [],
    "lr": [],
}

best_f1 = -1.0
best_epoch = -1
patience_counter = 0
best_ckpt_path = "best_model.pth"

for epoch in range(MAX_EPOCHS):
    if epoch == FREEZE_BACKBONE_EPOCHS:
        set_backbone_trainable(model, trainable=True)
        optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max(MAX_EPOCHS - epoch, 1))
        print("Backbone descongelado. Fine-tuning completo activado.")

    model.train()
    running_loss = 0.0

    for imgs, labels in train_loader:
        imgs = imgs.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=amp_enabled):
            logits = model(imgs)
            loss = criterion(logits, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        running_loss += loss.item() * imgs.size(0)

    scheduler.step()

    train_loss = running_loss / len(train_ds)
    metrics = evaluate_model(model, val_loader, device)

    history["train_loss"].append(train_loss)
    history["val_f1"].append(metrics["f1_macro"])
    history["val_spec"].append(metrics["spec_macro"])
    history["val_bal_acc"].append(metrics["bal_acc"])
    history["lr"].append(optimizer.param_groups[0]["lr"])

    print(
        f"Epoch {epoch+1:02d}/{MAX_EPOCHS} | "
        f"loss={train_loss:.4f} | "
        f"F1_macro={metrics['f1_macro']:.4f} | "
        f"Spec_macro={metrics['spec_macro']:.4f} | "
        f"BalAcc={metrics['bal_acc']:.4f} | "
        f"lr={optimizer.param_groups[0]['lr']:.6f}"
    )

    if metrics["f1_macro"] > best_f1:
        best_f1 = metrics["f1_macro"]
        best_epoch = epoch + 1
        patience_counter = 0

        torch.save(
            {
                "model_state_dict": model.state_dict(),
                "arch": "resnet18",
                "f1_val": float(metrics["f1_macro"]),
                "spec_val": float(metrics["spec_macro"]),
                "epoch": int(best_epoch),
                "class_names": CLASS_NAMES,
                "image_size": IMAGE_SIZE,
                "seed": SEED,
            },
            best_ckpt_path,
        )
        print(f"  -> Nuevo mejor checkpoint guardado en: {best_ckpt_path}")
    else:
        patience_counter += 1
        print(f"  -> Sin mejora. patience = {patience_counter}/{PATIENCE}")

    if patience_counter >= PATIENCE:
        print(f"\nEarly stopping activado en la época {epoch+1}.")
        break

print(f"\nMejor época: {best_epoch} | Mejor F1 Macro: {best_f1:.4f}")

In [ ]:
# ==========================================
# 11. Curvas de entrenamiento
# ==========================================
epochs_ran = len(history["train_loss"])
x = np.arange(1, epochs_ran + 1)

plt.figure(figsize=(8, 4))
plt.plot(x, history["train_loss"], marker="o")
plt.title("Train loss")
plt.xlabel("Época")
plt.ylabel("Loss")
plt.grid(True, alpha=0.3)
plt.show()

plt.figure(figsize=(8, 4))
plt.plot(x, history["val_f1"], marker="o", label="F1 Macro")
plt.plot(x, history["val_spec"], marker="o", label="Spec Macro")
plt.plot(x, history["val_bal_acc"], marker="o", label="Balanced Accuracy")
plt.title("Métricas de validación")
plt.xlabel("Época")
plt.ylabel("Score")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# ==========================================
# 12. Cargar mejor checkpoint y evaluar
# ==========================================
best_ckpt = torch.load(best_ckpt_path, map_location=device)
model.load_state_dict(best_ckpt["model_state_dict"])

val_metrics = evaluate_model(model, val_loader, device)

print("=== RESULTADOS EN VALIDACIÓN ===")
print(f"Best epoch      : {best_ckpt.get('epoch')}")
print(f"F1 Macro        : {val_metrics['f1_macro']:.4f}")
print(f"Spec Macro      : {val_metrics['spec_macro']:.4f}")
print(f"Balanced Acc    : {val_metrics['bal_acc']:.4f}")
print()

print("=== CLASSIFICATION REPORT ===")
print(classification_report(
    val_metrics["y_true"],
    val_metrics["y_pred"],
    labels=list(range(len(CLASS_NAMES))),
    target_names=CLASS_NAMES,
    digits=4,
    zero_division=0,
))

In [ ]:
plot_confusion(val_metrics["y_true"], val_metrics["y_pred"], normalize=False)
plot_confusion(val_metrics["y_true"], val_metrics["y_pred"], normalize=True)

## Párrafo breve para el README / notebook
Puedes dejar algo así y solo actualizar los números finales:

> Se utilizó una **ResNet-18 preentrenada en ImageNet** por ser una arquitectura ligera y adecuada para la restricción de **4 GB de VRAM**. Para mitigar el desbalance de clases se aplicó **WeightedRandomSampler** y una **Focal Loss con pesos por clase**. El mejor checkpoint se guardó según **F1 Macro en validación**, alcanzando un F1 Macro de **X.XXXX** y una Especificidad Macro de **Y.YYYY** antes del test ciego. Esta elección está alineada con la guía rápida de la rúbrica para el sprint. fileciteturn4file0

In [ ]:
# ==========================================
# 13. Exportar requirements.txt
# ==========================================
# La rúbrica pide entregar requirements.txt con versiones.
# Esta celda intenta generarlo desde el entorno actual.

req_path = "requirements.txt"

packages = [
    "torch",
    "torchvision",
    "numpy",
    "scikit-learn",
    "matplotlib",
    "Pillow",
]

try:
    result = subprocess.run(
        ["python", "-m", "pip", "freeze"],
        capture_output=True,
        text=True,
        check=True,
    )
    freeze_lines = result.stdout.splitlines()
    selected = []
    for p in packages:
        prefix = p.lower() + "=="
        match = next((line for line in freeze_lines if line.lower().startswith(prefix)), None)
        if match is not None:
            selected.append(match)

    with open(req_path, "w", encoding="utf-8") as f:
        f.write("\n".join(selected) + "\n")

    print(f"Archivo generado: {req_path}")
    print(open(req_path, "r", encoding="utf-8").read())
except Exception as e:
    print("No pude generar requirements.txt automáticamente:", e)
    print("Crea el archivo manualmente con pip freeze o pip list.")

## Evaluación oficial del test ciego
La rúbrica indica que al final debes ejecutar `evaluate_check.py` con `best_model.pth`, la arquitectura usada y el archivo `galaxy10_blind_test.npz`. fileciteturn4file0

In [ ]:
# ==========================================
# 14. Test ciego oficial (últimos 10 min)
# ==========================================
# Ejecuta esta celda SOLO cuando el instructor libere:
#   - galaxy10_blind_test.npz
#   - evaluate_check.py

# !python evaluate_check.py \
#     --model best_model.pth \
#     --arch resnet18 \
#     --test_data galaxy10_blind_test.npz \
#     --team "Nombre del Equipo" \
#     --output resultados/

## Observación importante
En la rúbrica hay una inconsistencia: en una parte aparecen umbrales altos para los niveles, pero en otra se mencionan metas más bajas como si fueran "sobresaliente". Para entrenar bien, usa como objetivo práctico maximizar **F1 Macro** y **Especificidad Macro** en validación y llegar con el checkpoint listo antes del test ciego. fileciteturn4file0